# SMS Spam Classifier — Learning by Building

**Goal:** train and compare text-classification models on the UCI SMS Spam Collection, evaluate them honestly, then export a model for a small web app.

## Learning outcomes
- Inspect real labelled text data with Pandas.
- Understand class imbalance and why accuracy alone can mislead.
- Learn the purpose of a train/test split.
- Convert text to numeric features using TF-IDF.
- Train a Multinomial Naive Bayes baseline and compare it with Logistic Regression.
- Evaluate precision, recall, F1-score and a confusion matrix.
- Save a complete preprocessing + model pipeline for deployment.

## Dataset and attribution
Dataset: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)  
Citation: Almeida, T. & Hidalgo, J. (2011). *SMS Spam Collection*. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84  
License: CC BY 4.0. Attribute the dataset authors and UCI when redistributing or showcasing this work.

**Scope note:** this is an SMS spam detector, not a validated financial-fraud detector. It can be a starting point for a later scam-message project, but that requires relevant labelled scam data and separate evaluation.

## 0. Setup

Google Colab usually has the common data-science packages preinstalled. Run this cell if any import is missing.

**Concept:** a notebook is an executable lab book. Keep explanations beside code and record observations after running each section.

In [ ]:
%pip -q install pandas scikit-learn matplotlib seaborn joblib

## 1. Imports and reproducibility

We use a fixed random seed so the train/test split is reproducible. Reproducibility matters when comparing models.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score
)
import joblib

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 120)

## 2. Download and load the real dataset

UCI provides a small ZIP archive containing a tab-separated file. The two columns are the label (`ham` or `spam`) and the message text.

**Why download from the source?** It makes the data provenance clear and avoids relying on an unknown re-upload. If the download fails in Colab, download the ZIP from the UCI page and upload it manually.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

with urllib.request.urlopen(DATA_URL, timeout=30) as response:
    archive_bytes = response.read()

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as zf:
    print("Files in archive:", zf.namelist())
    raw = zf.read("SMSSpamCollection")

df = pd.read_csv(
    io.BytesIO(raw),
    sep="\t",
    header=None,
    names=["label", "text"],
    encoding="utf-8"
)

print("Shape:", df.shape)
display(df.head(10))

### Observe before cleaning

Look at the shape, column types, missing values and duplicate rows before changing the data. Cleaning without inspecting can accidentally remove useful information.

In [ ]:
print("Data types:")
display(df.dtypes)

print("\\nMissing values:")
display(df.isna().sum())

print("\\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate message texts:", df["text"].duplicated().sum())
print("\\nLabels:", sorted(df["label"].unique()))

## 3. Data quality and class distribution

`ham` means a normal message; `spam` means a message labelled as spam in this dataset.

**Class imbalance:** datasets often contain more normal messages than spam. A model that predicts every message as normal could get deceptively high accuracy, so later we will inspect spam precision and recall too.

In [ ]:
df = df.dropna(subset=["label", "text"]).copy()
df["label"] = df["label"].str.strip().str.lower()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["label"].isin(["ham", "spam"])]
df = df[df["text"].ne("")]
df = df.drop_duplicates(subset=["label", "text"]).reset_index(drop=True)

display(df["label"].value_counts().rename("count").to_frame())
display((df["label"].value_counts(normalize=True) * 100).round(2).rename("percent").to_frame())

ax = sns.countplot(data=df, x="label")
ax.set_title("Class distribution")
ax.set_xlabel("Message label")
ax.set_ylabel("Number of messages")
plt.show()

### Mini-checkpoint
Write down:
1. How many rows remain after cleaning?
2. Which class is more common?
3. Why could accuracy alone hide a poor spam detector?

## 4. Define features (`X`) and target (`y`)

- **Feature (`X`):** the message text the model can use.
- **Target (`y`):** the correct label we want it to predict.

We keep the original message text. TF-IDF will convert it to numbers inside the pipeline.

In [ ]:
X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\\nTraining class proportions:")
display(y_train.value_counts(normalize=True).round(3))
print("\\nTest class proportions:")
display(y_test.value_counts(normalize=True).round(3))

## 5. Text → numbers: TF-IDF

Machine-learning algorithms cannot directly learn from raw strings. `TfidfVectorizer` builds a vocabulary and assigns numeric weights to terms. Roughly, terms are more informative when they appear often in a message but are not common across every message.

**Leakage prevention:** the vectorizer is part of a scikit-learn `Pipeline`. During `.fit()`, its vocabulary and IDF weights are learned from training messages only. Test messages are transformed using those learned values; they do not influence training.

In [ ]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("model", MultinomialNB())
])

nb_pipeline.fit(X_train, y_train)
print("Naive Bayes pipeline trained.")
print("Vocabulary size:", len(nb_pipeline.named_steps["tfidf"].vocabulary_))

## 6. Train a comparison model

Naive Bayes is a fast, strong baseline for word-count/TF-IDF text classification. Logistic Regression is another widely used linear classifier for sparse text features.

We compare both on the **same untouched test set**. We choose the final model by spam-class F1 in this teaching example, but a real deployment should choose a metric and operating threshold based on the cost of false positives vs false negatives.

In [ ]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

lr_pipeline.fit(X_train, y_train)
print("Logistic Regression pipeline trained.")

## 7. Evaluate with more than accuracy

- **Precision (spam):** among messages predicted as spam, how many really are spam?
- **Recall (spam):** among all actual spam messages, how many did we catch?
- **F1-score:** harmonic mean of precision and recall.
- **Confusion matrix:** shows correct and incorrect predictions by class.

For a spam detector, low recall means more spam slips through; low precision means more legitimate messages are wrongly flagged.

In [ ]:
models = {
    "Multinomial Naive Bayes": nb_pipeline,
    "Logistic Regression": lr_pipeline
}

rows = []
reports = {}
for name, model in models.items():
    pred = model.predict(X_test)
    reports[name] = classification_report(y_test, pred, output_dict=True, zero_division=0)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "spam_precision": precision_score(y_test, pred, pos_label="spam", zero_division=0),
        "spam_recall": recall_score(y_test, pred, pos_label="spam", zero_division=0),
        "spam_f1": f1_score(y_test, pred, pos_label="spam", zero_division=0)
    })

results = pd.DataFrame(rows).sort_values("spam_f1", ascending=False).reset_index(drop=True)
display(results.style.format({
    "accuracy": "{:.3f}",
    "spam_precision": "{:.3f}",
    "spam_recall": "{:.3f}",
    "spam_f1": "{:.3f}"
}))

In [ ]:
for name, model in models.items():
    print("=" * 80)
    print(name)
    print(classification_report(y_test, model.predict(X_test), digits=3, zero_division=0))

In [ ]:
best_name = results.loc[0, "model"]
best_model = models[best_name]
best_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, best_pred, labels=["ham", "spam"])
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted ham", "Predicted spam"],
    yticklabels=["Actual ham", "Actual spam"]
)
plt.title(f"Confusion matrix — {best_name}")
plt.ylabel("Actual label")
plt.xlabel("Predicted label")
plt.show()

print("Selected model:", best_name)

## 8. Inspect errors, not just scores

False positives are legitimate messages classified as spam. False negatives are spam messages classified as legitimate. Reading examples often reveals what the model has learned and where it fails.

Do not use this small test set to keep tweaking the model repeatedly and then report it as an unbiased final evaluation. For a larger project, use validation data or cross-validation for model selection and reserve a final test set.

In [ ]:
error_df = pd.DataFrame({
    "text": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(best_pred)
})
errors = error_df[error_df["actual"] != error_df["predicted"]]
display(errors.head(20))
print("Total test errors:", len(errors))

## 9. Try messages the model has never seen

These are hand-written smoke tests, not a formal evaluation set. Predictions may vary by model and should not be treated as proof that a message is safe.

In [ ]:
demo_messages = [
    "Congratulations! You won a free prize. Call now to claim.",
    "Hey, are we still meeting at the library at 5?",
    "URGENT! Your account is blocked. Click this link immediately.",
    "Can you send me the notes from today's class?"
]

for message in demo_messages:
    prediction = best_model.predict([message])[0]
    print(f"Prediction: {prediction.upper():4} | {message}")

## 10. Export a deployment artifact

The saved object is the **whole pipeline** (TF-IDF + classifier), not only the classifier. This is important: the deployed app must apply exactly the same text-to-feature transformation used during training.

We save:
- `spam_classifier.joblib`: fitted pipeline
- `metrics.json`: measured test metrics
- `test_predictions.csv`: test labels and predictions for inspection

Only load joblib/pickle files you created or trust; loading an untrusted serialized file can execute malicious code.

In [ ]:
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(best_model, OUTPUT_DIR / "spam_classifier.joblib")

metrics_payload = {
    "selected_model": best_name,
    "test_size": int(len(X_test)),
    "accuracy": float(accuracy_score(y_test, best_pred)),
    "spam_precision": float(precision_score(y_test, best_pred, pos_label="spam", zero_division=0)),
    "spam_recall": float(recall_score(y_test, best_pred, pos_label="spam", zero_division=0)),
    "spam_f1": float(f1_score(y_test, best_pred, pos_label="spam", zero_division=0)),
    "random_state": RANDOM_STATE,
    "dataset": "UCI SMS Spam Collection",
    "dataset_doi": "10.24432/C5CC84"
}

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    import json
    json.dump(metrics_payload, f, indent=2)

error_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
print("Saved artifacts:")
for path in OUTPUT_DIR.iterdir():
    print(path, path.stat().st_size, "bytes")

## 11. Download the model for the web app

In Colab's file panel, open `artifacts/`, select `spam_classifier.joblib`, and download it. Also download `metrics.json` if you want the deployed app to show your measured test metrics.

Place the model file beside `app.py` in the deployment repository. Do not invent performance numbers in your README; copy the values from the output of this notebook.

In [ ]:
from google.colab import files
files.download("artifacts/spam_classifier.joblib")
files.download("artifacts/metrics.json")

## 12. Final reflection

Before calling this project complete, answer in your own words:
1. Why do we split data before fitting TF-IDF?
2. When might recall matter more than precision?
3. What does each cell in the confusion matrix represent?
4. What kinds of spam could this dataset fail to represent?
5. Why is this not yet a financial-fraud classifier?
6. What new data and evaluation would be needed for Telugu-English scam messages?

## Further work
- Add a validation split and tune the decision threshold.
- Compare word n-grams with character n-grams.
- Test for duplicates and near-duplicates across splits.
- Add a FastAPI JSON endpoint if a separate backend is needed.
- For fraud awareness, obtain a task-appropriate, legally usable labelled dataset and evaluate on that domain separately.